# Ply pipeline on Kaggle (T4)

Phone walkthrough video → COLMAP 4.2.1 poses → gsplat (MCMC + PPISP) → `scene.spz` + floor/occupancy for the web walkthrough.

**Before running**
1. *Settings → Accelerator → GPU T4 x1* (P100 is **not** supported by torch cu128) and *Internet on*.
2. *Add input*:
   - your capture video as a Dataset (any name; the first `.mp4/.mov/.MOV` found under `/kaggle/input` is used),
   - the `pipeline/` folder of the Ply repo uploaded as a Dataset named **`ply-pipeline`**,
   - optional: a Dataset **`ply-wheels`** made from a previous run's `/kaggle/working/wheels` (saves ~15 min of CUDA builds).
3. *Run all.* Outputs land in `/kaggle/working/output` (and `output.zip`).

Every stage is resumable: if the session dies, re-run all and finished stages are skipped (as long as `/kaggle/working/work` survived, e.g. via *Save Version → Quick save* outputs).

In [ ]:
# ---- settings ----
SCENE = "capture"          # output folder name
PRESET = "standard"        # quick (~25 min on T4) | standard (~1.5-2 h) | high (needs > 16 GB: use RunPod)
HEIGHT_CAM = 1.45          # how high the phone was held, metres (sets the metric scale)
EXTRA_ARGS = []            # e.g. ["--max-frames", "400"] or ["--features", "sift"]
VIDEO = None               # None = first video found under /kaggle/input

In [ ]:
import glob, os, subprocess, sys, json, shutil, pathlib
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
assert gpu, "No GPU: enable Accelerator -> GPU T4 x1"
if "P100" in gpu:
    raise SystemExit("P100 (sm_60) is not supported by torch 2.11 cu128 wheels. Switch the accelerator to T4.")
ARCH = "7.5" if "T4" in gpu else ("8.9" if ("L4" in gpu or "4090" in gpu) else "7.5;8.0;8.6;8.9")
print("TORCH_CUDA_ARCH_LIST =", ARCH, "| python", sys.version.split()[0])

In [ ]:
# pipeline source -> /kaggle/working/pipeline (writable, editable install)
def find_input(rel, must=None):
    """Kaggle has moved dataset mounts before (/kaggle/input/<slug>/ vs nested owner folders); search instead of assuming."""
    hits = [p for p in glob.glob(f"/kaggle/input/**/{rel}", recursive=True) if must is None or must(p)]
    if not hits:
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
            if depth <= 3:
                print("  ", root, sorted(files)[:8])
    return sorted(hits, key=len)[0] if hits else None
src = find_input("pyproject.toml", lambda p: os.path.isdir(os.path.join(os.path.dirname(p), "plypipe")))
assert src, "Add the Ply repo's pipeline/ folder as a Dataset named ply-pipeline"
PIPE = "/kaggle/working/pipeline"
if not os.path.exists(PIPE):
    shutil.copytree(os.path.dirname(src), PIPE, ignore=shutil.ignore_patterns(".venv", "work", "__pycache__"))
wheels = find_input("gsplat-*.whl")
env = dict(os.environ, TORCH_CUDA_ARCH_LIST=ARCH, PLY_GSPLAT_DIR="/kaggle/working/gsplat", MAX_JOBS="4", PYTHON=sys.executable)
if wheels:
    env["PLY_WHEELS"] = os.path.dirname(wheels); print("using prebuilt wheels from", env["PLY_WHEELS"])
else:
    env["PLY_SAVE_WHEELS"] = "/kaggle/working/wheels"; print("building CUDA wheels; they will be saved to /kaggle/working/wheels")
print("nvcc:", subprocess.run(["bash", "-c", "nvcc --version | tail -2"], capture_output=True, text=True).stdout.strip())
with open("/kaggle/working/install.log", "w") as f:  # full log kept as an output file; the tail is printed if it fails
    r = subprocess.run(["bash", f"{PIPE}/scripts/install_gpu.sh"], env=env, stdout=f, stderr=subprocess.STDOUT)
log_tail = open("/kaggle/working/install.log").read()[-6000:]
print(log_tail if r.returncode else log_tail[-800:])
assert r.returncode == 0, "install failed: see the log above (/kaggle/working/install.log)"

In [ ]:
# Node.js for splat-transform (SPZ export)
if shutil.which("npx") is None:
    subprocess.run("curl -fsSL https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz | tar -xJ -C /kaggle/working", shell=True, check=True)
    os.environ["PATH"] = "/kaggle/working/node-v22.14.0-linux-x64/bin:" + os.environ["PATH"]
print(subprocess.run(["node", "-v"], capture_output=True, text=True).stdout)

In [ ]:
if VIDEO is None:
    vids = sorted(p for ext in ("mp4", "mov", "MOV", "MP4", "m4v") for p in glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True))
    assert vids, "No video found under /kaggle/input"
    VIDEO = vids[0]
print("video:", VIDEO)
OUT = f"/kaggle/working/work/{SCENE}"
cmd = [sys.executable, "-m", "plypipe", "run", VIDEO, "--out", OUT, "--preset", PRESET, "--height-cam", str(HEIGHT_CAM),
       "--gsplat-dir", "/kaggle/working/gsplat", *EXTRA_ARGS]
print(" ".join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=PIPE)
for line in p.stdout:
    try:
        e = json.loads(line)
        if e["event"] in ("start", "done", "skip", "warn", "error") or e["stage"] != "train":
            print(f'[{e["stage"]}] {e["event"]} {e.get("msg", "")}', {k: v for k, v in e.items() if k not in ("t", "stage", "event", "msg", "trace")} or "")
        if e["event"] == "error":
            print(e.get("trace", ""))
    except Exception:
        print(line, end="")
assert p.wait() == 0, "pipeline failed - see the error above and work/<scene>/logs/"

In [ ]:
# collect what the web viewer needs
dst = pathlib.Path("/kaggle/working/output"); dst.mkdir(exist_ok=True)
for f in ("scene.spz", "scene_lite.spz", "scene.json", "occupancy.png", "occupancy.json", "report.json"):
    shutil.copy2(f"{OUT}/{f}", dst / f)
shutil.make_archive("/kaggle/working/output", "zip", dst)
print(json.dumps(json.load(open(dst / "report.json")), indent=1)[:3000])
print(sorted(os.listdir(dst)))

In [ ]:
# quick look: occupancy grid (white = walkable, black = blocked, grey = unknown) with the capture path
import matplotlib.pyplot as plt, numpy as np
from PIL import Image
occ = np.array(Image.open(dst / "occupancy.png")); meta = json.load(open(dst / "occupancy.json")); scene = json.load(open(dst / "scene.json"))
path = np.array(scene["cameraPath"]); cs = meta["cellSize"]; ox, oz = meta["origin"]
plt.figure(figsize=(8, 8)); plt.imshow(occ, cmap="gray", vmin=0, vmax=255)
plt.plot((path[:, 0] - ox) / cs, (path[:, 2] - oz) / cs, "-", color="#FF6B00", lw=2); plt.title("occupancy + capture path"); plt.axis("equal"); plt.show()